In [1]:
import numpy as np, pandas as pd
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.functions import vector_to_array

spark = SparkSession.builder.master("local[*]").appName("ews-explain").getOrCreate()
CUTOFF, THRESH = 28, 0.38
cat_cols = ["code_module", "gender", "region", "highest_education",
            "imd_band", "age_band", "disability"]
meta = set(cat_cols + ["id_student", "code_presentation", "at_risk", "split"])

df = spark.read.parquet(f"../data/processed/features_day{CUTOFF}.parquet")
num_cols = [c for c in df.columns if c not in meta]
idx = StringIndexer(inputCols=cat_cols, outputCols=[c + "_i" for c in cat_cols], handleInvalid="keep")
ohe = OneHotEncoder(inputCols=[c + "_i" for c in cat_cols], outputCols=[c + "_v" for c in cat_cols])
vec = VectorAssembler(inputCols=num_cols + [c + "_v" for c in cat_cols], outputCol="features")
model = Pipeline(stages=[idx, ohe, vec, LogisticRegression(labelCol="at_risk", maxIter=100)]) \
        .fit(df.filter("split = 'train'"))

scored = model.transform(df)
coef = model.stages[-1].coefficients.toArray()

# feature names from the assembler's metadata
attrs = scored.schema["features"].metadata["ml_attr"]["attrs"]
names = [None] * len(coef)
for group in attrs.values():
    for a in group:
        names[a["idx"]] = a["name"]

pdf = scored.select("id_student", "code_module", "code_presentation", "split", "at_risk",
                    vector_to_array("features").alias("x"),
                    vector_to_array("probability")[1].alias("risk")).toPandas()
X = np.vstack(pdf.x.values)
mu = X[pdf.split == "train"].mean(axis=0)
contrib = (X - mu) * coef            # log-odds contribution of each feature

# sanity check: base + contributions should equal each student's log-odds
base = model.stages[-1].intercept + coef @ mu
logit = np.log(pdf.risk / (1 - pdf.risk))
print("max error:", np.abs(base + contrib.sum(axis=1) - logit).max())

# look at one flagged student
i = pdf.index[pdf.risk >= THRESH][0]
top = np.argsort(-contrib[i])[:5]
print(pdf.loc[i, ["id_student", "risk"]].to_dict())
for j in top:
    print(f"  {names[j]:30s} value={X[i, j]:8.2f}  impact={contrib[i, j]:+.3f}")

max error: 7.638334409421077e-14
{'id_student': 106577, 'risk': 0.4584407960132926}
  no_submission                  value=    0.00  impact=+0.304
  highest_education_v_Lower Than A Level value=    1.00  impact=+0.216
  imd_band_v_20-30%              value=    1.00  impact=+0.168
  active_days                    value=   12.00  impact=+0.157
  code_module_v_GGG              value=    0.00  impact=+0.114


In [2]:
def group_of(n):
    if n.startswith("code_module"):
        return "course"
    if n.startswith(("gender", "region", "highest_education", "imd_band", "age_band",
                     "disability", "studied_credits", "num_of_prev_attempts")):
        return "background"
    if n in ("no_submission", "avg_score", "n_submitted", "n_missed", "n_late", "n_due"):
        return "assessments"
    return "engagement"

for n in ["no_submission", "avg_score", "n_submitted"]:
    j = names.index(n)
    print(f"{n:15s} weight={coef[j]:+.3f}  this student's impact={contrib[i, j]:+.3f}")

groups = np.array([group_of(n) for n in names])
G = pd.DataFrame({g: contrib[:, groups == g].sum(axis=1) for g in np.unique(groups)})

print("\nexample student, by theme:")
print(G.loc[i].round(3).to_dict())
print("sum of themes:", round(G.loc[i].sum(), 3), "| sum of all features:", round(contrib[i].sum(), 3))

flagged = (pdf.risk >= THRESH).values
print("\naverage push toward risk among flagged students, by theme:")
print(G[flagged].mean().round(3).sort_values(ascending=False))

no_submission   weight=-1.188  this student's impact=+0.304
avg_score       weight=-0.021  this student's impact=-0.054
n_submitted     weight=-0.019  this student's impact=-0.002

example student, by theme:
{np.str_('assessments'): -0.14, np.str_('background'): 0.439, np.str_('course'): -0.316, np.str_('engagement'): 0.102}
sum of themes: 0.085 | sum of all features: 0.085

average push toward risk among flagged students, by theme:
assessments    0.331
engagement     0.330
background     0.191
course         0.058
dtype: float64
